In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_002 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_002_cat_strict_raw_baseline.npy"
)
oof_006b = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy"
)
oof_006c = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006c_cat_original_prior_no_count_min.npy"
)
oof_007 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_007_realmlp_shared001_min.npy"
)

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("002_cat_raw", oof_002)
report("006b_cat_original_prior_no_year", oof_006b)
report("006c_cat_original_prior_no_count", oof_006c)
report("007_realmlp_shared001_min", oof_007)

pairs = [
    ("002", oof_002, "007", oof_007),
    ("006b", oof_006b, "007", oof_007),
    ("006c", oof_006c, "007", oof_007),
]

print("\nCorrelation:")
for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

blend_candidates = {
    "avg_006b_007": (oof_006b + oof_007) / 2,
    "avg_006c_007": (oof_006c + oof_007) / 2,
    "avg_002_006b_007": (oof_002 + oof_006b + oof_007) / 3,
    "avg_002_006b_006c_007": (oof_002 + oof_006b + oof_006c + oof_007) / 4,
}

print("\nBlend probe:")
for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))

002_cat_raw
  AUC: 0.9485020378244504

006b_cat_original_prior_no_year
  AUC: 0.9487943937294769

006c_cat_original_prior_no_count
  AUC: 0.9485750913142212

007_realmlp_shared001_min
  AUC: 0.9532701289015783


Correlation:
002 vs 007: pearson=0.973084, spearman=0.964428
006b vs 007: pearson=0.974165, spearman=0.963871
006c vs 007: pearson=0.973444, spearman=0.963171

Blend probe:
avg_006b_007 0.9528554748055666
avg_006c_007 0.9527959525088343
avg_002_006b_007 0.9520215066352702
avg_002_006b_006c_007 0.9514334100505152
